In [ ]:
# importing and installing essential libraries
import datasets
import transformers
import accelerate
try:
  import torchmetrics
  import gradio as gr
except Exception as e:
  ! pip install torchmetrics
  ! pip install gradio
  import gradio as gr
  import torchmetrics
print(f'[INFO] Transformer :{transformers.__version__}')
print(f'[INFO] datasets :{datasets.__version__}')
print(f'[INFO] accelerate :{accelerate.__version__}')
print(f'[INFO] torchmetrics :{torchmetrics.__version__}')
print(f'[INFO] gradio :{gr.__version__}')

In [ ]:
## loading dataset
from datasets import load_dataset
dataset = datasets.load_dataset("mrdbourke/trashify_manual_labelled_images")
# dataset is in coco format

In [ ]:
dataset["train"][42] # seeing random index

In [ ]:
## plotting random image
dataset["train"][42]["image"]
# so from this their is no need to get any preprocess function to make the image in half with bounding box

In [ ]:
## creating labels (label2id and id2label)
categories = dataset["train"].features["annotations"]["category_id"]
id2label = {key:val for key,val in enumerate(categories.feature.names)}
label2id = {val:key for key,val in id2label.items()}

In [ ]:
# define the color palette
color_palette = {
    'bin': (0, 0, 224),         # Bright Blue (High contrast with greenery) in format (red, green, blue)
    'not_bin': (255, 80, 80),   # Light Red to indicate negative class

    'hand': (148, 0, 211),      # Dark Purple (Contrasts well with skin tones)
    'not_hand': (255, 80, 80),  # Light Red to indicate negative class

    'trash': (0, 255, 0),       # Bright Green (For trash-related items)
    'not_trash': (255, 80, 80), # Light Red to indicate negative class

    'trash_arm': (255, 140, 0), # Deep Orange (Highly visible)
}

def normalize_color_pal(color_code):
  return tuple(x/255.0 for x in color_code)

In [ ]:
# creating and random function to see how the data is and also displaying with bounding box
import random
import torch
import torchvision.ops as ops # to convert the boxes to required format
from torchvision.utils import draw_bounding_boxes # to plot bounding box with image
from torchvision.transforms.functional import pil_to_tensor , to_pil_image# to convert pil to tensor
def rand_index(num_rows):
  random_index = random.randint(0,num_rows-1)
  return random_index

def create_sample(dataset):
  random_index =  rand_index(dataset["train"].num_rows) # to get the random_index
  random_sample = dataset["train"][random_index] # random sample
  print(f'[INFO] Random index:{random_index}')
  # random_sample : imp features (bbox,category,image,image_id[optional])
  bbox = torch.Tensor(random_sample["annotations"]["bbox"]) # it is list of bboxes , (xywh) format
  print(f"[INFO] Bounding boxes:{bbox}")
  image = random_sample["image"] # the PIL image
  category = random_sample["annotations"]["category_id"]
  print(f'[INFO] Category id : {[cat for cat in category]}')
  image_id = random_sample["image_id"] # images id == random_index
  bbox_xyxy = ops.box_convert(bbox,in_fmt="xywh",out_fmt="xyxy") # to convert
  # print(bbox_xyxy)
  labels_sample = [id2label[x] for x in category] # labels
  print(f'[INFO] Labels :{labels_sample}')
  color_pal_sample = [tuple(int(c) for c in color_palette[x]) for x in labels_sample]
  # color pal for labels
  print(f'[INFO] color palette :{color_pal_sample}')
  return to_pil_image(
    pic = draw_bounding_boxes(
          image = pil_to_tensor(
              image
          ).to(torch.uint8), # just to reduce the size
          boxes = bbox_xyxy,
          colors = color_pal_sample,
          labels= labels_sample,
          label_colors = color_pal_sample,
      )
  )

In [ ]:
create_sample(dataset)
# okay so step 1 : complete to create the view of dataset

In [ ]:
## now lets work with image processor

In [ ]:
## Defining model name [GLOBAL]
MODEL_NAME = "PekingU/rtdetr_v2_r50vd"

In [ ]:
from transformers import AutoImageProcessor

In [ ]:
image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path = MODEL_NAME,
    return_tensors="pt",
)

In [ ]:
image_processor # the original one (default)

In [ ]:
# so the above is the image processor (for the rtdetr model) and we can edit that things according to our type of dataset
IMAGE_WIDTH = 640
IMAGE_HEIGHT = 640
image_processor = AutoImageProcessor.from_pretrained(
    pretrained_model_name_or_path = MODEL_NAME,
    return_tensors = "pt",
    do_convert_annotations = True,
    format = "coco_detection",
    size = {
        "shortest_edge": IMAGE_HEIGHT,
        "longest_edge": IMAGE_WIDTH
    },
    return_segmentation_masks = True,
    do_pad = True,
    use_fast = True
)

In [ ]:
image_processor # the modified one

In [ ]:
## This is the standard COCO format used : And we want to take data approx. equal to it
# # COCO format, see: https://cocodataset.org/#format-data
# [{
#     "image_id": 42,
#     "annotations": [{
#         "id": 123456,
#         "category_id": 1,
#         "iscrowd": 0,
#         "segmentation": [
#             [42.0, 55.6, ... 99.3, 102.3]
#         ],
#         "image_id": 42, # this matches the 'image_id' field above
#         "area": 135381.07,
#         "bbox": [523.70,
#                  545.09,
#                  402.79,
#                  336.11]
#     },
#     # Next annotation in the same format as the previous one (one annotation per dict).
#     # For example, if an image had 4 bounding boxes, there would be a list of 4 dictionaries
#     # each containing a single annotation.
#     ...]
# }]

In [ ]:
dataset["train"][42] # random sample to compare the coco format annotations

In [ ]:
! pip install typing dataclasses

In [ ]:
## so the annotations (coco) are not in place so we want to modify it
from typing import List,Dict
from dataclasses import dataclass , asdict # to create and type int class

In [ ]:
@dataclass
class SingleImageAnnotation: # this is for single category , return type dict
  image_id : int
  area : float
  bbox : List[float]
  category_id : int
  is_crowd : int = 0

  def __post_init__(self): # so this is automatically called when dataclass is generated
    if self.bbox.__len__()!=4:
      raise ValueError("[INFO] Bounding Box length not matched")

In [ ]:
@dataclass
class COCOImageAnnotations: # to this is for the image with collection of category wise annotations
  image_id : int
  annotations : List[SingleImageAnnotation]

In [ ]:
# so to do this modifications in our data to match the model training set
## we will create an function which take category_id[list],area[list],bbox[list(float)],image_id to get the final coco format data

In [ ]:
## It is just the testing code for  building and preprocess function
random_sample = dataset["train"][42]
annotations = SingleImageAnnotation(
    image_id = random_sample["image_id"],
    area =  random_sample["annotations"]["area"],
    bbox =  random_sample["annotations"]["bbox"][0],
    category_id = random_sample["annotations"]["category_id"]
)
List[asdict(annotations)] # for single cateogry

In [ ]:
COCOImageAnnotations(
    image_id = random_sample["image_id"],
    annotations = List[asdict(annotations)]
)

In [ ]:
def format_coco_annotations(
    image_id : int,
    categories : List[int],
    bboxes : List[list[float,float,float,float]],
    areas : List[float]
):
  """function which take category_id[list],area[list],bbox[list(float)],image_id to get the final coco format data """
  annotations = [
      asdict(SingleImageAnnotation(
          image_id = image_id,
          area = area,
          bbox = bbox,
          category_id = category_id
      ))
      for category_id,area,bbox in zip(categories,areas,bboxes)
  ]
  image_annotations = asdict(COCOImageAnnotations(
      image_id = image_id,
      annotations = annotations
  ))
  return image_annotations

In [ ]:
format_coco_annotations?

In [ ]:
image_id = random_sample["image_id"]
areas = random_sample["annotations"]["area"]
bboxes = random_sample["annotations"]["bbox"]
categories = random_sample["annotations"]["category_id"]
formated_ = format_coco_annotations(
    image_id = image_id,
    categories = categories,
    bboxes = bboxes,
    areas = areas
)
formated_
## Data for image processor (format) done !!

In [ ]:
preprocessed_input = image_processor.preprocess(
    images = random_sample["image"],
    annotations = formated_,
    return_tensors = "pt"
)
preprocessed_input

In [ ]:
preprocessed_input["labels"][0]["orig_size"]

In [ ]:
## loading the model
from transformers import AutoModelForObjectDetection
model = AutoModelForObjectDetection.from_pretrained(
    pretrained_model_name_or_path =  MODEL_NAME,
    label2id = label2id,
    id2label = id2label,
    ignore_mismatched_sizes=True,
)

In [ ]:
def create_model():
  ## just for reproduciablitiy
  model = AutoModelForObjectDetection.from_pretrained(
      pretrained_model_name_or_path = MODEL_NAME,
      label2id = label2id,
      id2label = id2label,
      ignore_mismatched_sizes=True, # default
  )
  return model

In [ ]:
model = create_model()
# okay to some weights are not trained just because it is matching outclass

In [ ]:
## creating and inference to check the preprocessed output is working or not
outputs = model(
    pixel_values = preprocessed_input["pixel_values"],
    pixel_mask = preprocessed_input["pixel_mask"]
)

In [ ]:
outputs.keys() # the raw logits

In [ ]:
## post processed outputs (using image processer)
THRESHOLD = 0.35
post_processed_outputs = image_processor.post_process_object_detection(
    outputs = outputs,
    threshold = THRESHOLD,
    target_sizes = preprocessed_input["labels"][0]["orig_size"].unsqueeze(0) # to match batch dimension what is required to process (model)
)

In [ ]:
post_processed_outputs # to this preprocess is handled internally by image_processor

In [ ]:
for score, label, box in zip(post_processed_outputs[0]["scores"], post_processed_outputs[0]["labels"], post_processed_outputs[0]["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    print(f"Detected {model.config.id2label[label.item()]} with confidence {round(score.item(), 3)} at location {box}")

In [ ]:
## moving to dataset split
split = dataset["train"].train_test_split(test_size=0.2)

dataset["train"] = dataset["train"]
split2 = split["test"].train_test_split(test_size=0.2)
dataset["test"] = split2["train"]
dataset["validation"] = split2["test"] # to make model more robust

In [ ]:
##checking does it worked
dataset

In [ ]:
## Creating and batch of data and observe wwther all things will work great
sample = dataset["train"][0:3]
sample

In [ ]:
dataset["train"][42]

In [ ]:
## okay so looking the sample it's completely differnet than
format_coco_annotations?

In [ ]:
for image_id,annotations,image in zip(sample["image_id"],sample["annotations"],sample["image"]):
  bbox = annotations["bbox"]
  category_id = annotations["category_id"]
  area = annotations ["area"]
  annotations = format_coco_annotations(image_id,category_id,bbox,area)
  processed = image_processor.preprocess(images=image,annotations=annotations,return_tensors="pt")
  # this processed is what we want to return and it get its that place
  break

In [ ]:
processed

In [ ]:
def preprocess_batch(batch,image_processor,transforms=None):
  images = []
  annotations_ = [] # for batch
  for image_id,annotations,image in zip(batch["image_id"],batch["annotations"],batch["image"]):
    bbox = annotations ["bbox"]
    category_id = annotations ["category_id"]
    area = annotations ["area"]
    coco_formated = format_coco_annotations(image_id,category_id,bbox,area)
    images.append(image)
    annotations_.append(coco_formated)
  processed_batch = image_processor.preprocess(images=images,annotations=annotations_,return_tensors="pt")
  return processed_batch

In [ ]:
preprocess_batch(sample,image_processor)

In [ ]:
## now creating and formalized processing class
from functools import partial

partial_preprocess = partial(preprocess_batch,image_processor=image_processor,transforms=None)
partial_preprocess

In [ ]:
processed_dataset = dataset.copy()

processed_dataset["train"] = dataset["train"].with_transform(transform = partial_preprocess)
processed_dataset["test"] = dataset["test"].with_transform(transform = partial_preprocess)
processed_dataset["validation"] = dataset["validation"].with_transform(transform = partial_preprocess)
# the preprocessing of that dataset is done i.e taking to annotations to pixel values and pixel masks which are input for model

In [ ]:
processed_dataset["train"][42] # checking random sample

In [ ]:
## creating and Collation function >> essential to stack all the dataset pixel values or mask as in batch format

In [ ]:
if "pixel_mask" in processed_dataset["train"][42]:
  print("h")

In [ ]:
from typing import List,Dict,Any

def collation_function(batch:List[Dict[str,Any]]):
  collated_data = {} # the dict that store the pixel value, mask , label in batch
  collated_data["pixel_values"] = torch.stack([sample["pixel_values"] for sample in batch])
  collated_data["labels"] = [sample["labels"] for sample in batch] # here it is already dict so no need
  if "pixel_mask" in batch[0]: # as batch is list
    collated_data["pixel_mask"] = torch.stack([sample["pixel_mask"] for sample in batch])  # this because some image processor not give pixel mask
  return collated_data

In [ ]:
%%time
# checking will it works
example_collated_data_batch = collation_function(processed_dataset["train"].select(range(32)))
example_collated_data_batch.keys()

In [ ]:
from pathlib import Path
model_path = Path("/content/checkpoints")
model_path.mkdir(exist_ok=True) # this directory is use to store the model checkpoints

In [ ]:
## setting learning rate to model parameter according to paper and optimizer
from transformers import Trainer

# Create lists for different kinds of parameters
backbone_parameters = []
other_parameters = []

# Can loop through model parameters and extract different model sections
for name, param in model.model.named_parameters():
    if "backbone" in name:
        # print(f"Backbone parameter: {name}")
        backbone_parameters.append(param)
    else:
        # print(f"Other parameter: {name}")
        other_parameters.append(param)

print(f"[INFO] Number of backbone parameter modules: {len(backbone_parameters)}")
print(f"[INFO] Number of other parameter modules: {len(other_parameters)}")

BACKBONE_LEARNING_RATE = 1e-5
DETECTION_HEAD_LEARNING_RATE = 1e-4

print(f"[INFO] Using learning rate for backbone: {BACKBONE_LEARNING_RATE}")
print(f"[INFO] Using learning rate for other parameters and detection head: {DETECTION_HEAD_LEARNING_RATE}")

# Setup a custom subclass of Trainer to use different learning rates for different parts of the model
class CustomTrainer(Trainer):
    def create_optimizer(self):
        self.optimizer = torch.optim.AdamW([
            {"params": backbone_parameters, "lr": BACKBONE_LEARNING_RATE},
            {"params": other_parameters, "lr": DETECTION_HEAD_LEARNING_RATE}
        ], weight_decay=0.0001)
        return self.optimizer

In [ ]:
## now all is set moving to training (creating trainer arguments)
from transformers import TrainingArguments

## setting some constants
BATCH_SIZE = 8
NUM_EPOCHS = 10
DATALOADER_NUM_WORKERS = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY =  1e-4
MAX_GRAD_NORM = 0.1 # gradient should not go above it
WARMUP_RATIO = 0.05 # total number of steps to take berfore reach to (1e-4) learning rate

# creating version file in model directory
OUTPUT_DIR = Path(model_path, "rt_detrv2_finetuned_drone_detection_v1")
print(f"[INFO] Saving model to: {OUTPUT_DIR}")
# creating trainer object
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    num_train_epochs=NUM_EPOCHS,
    lr_scheduler_type="linear",
    warmup_ratio=WARMUP_RATIO,
    # warmup_steps=2000, # number of warmup steps from 0 to learning_rate (overrides warmup_ratio, found this to be too long for our dataset)
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=True, # use mixed precision training
    dataloader_num_workers=DATALOADER_NUM_WORKERS, # note: if you're on Google Colab, you may have to lower this to os.cpu_count() or to 0
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False, # want to minimize eval_loss (e.g. lower is better)
    report_to="none", # don't save experiments to a third party service
    push_to_hub=False,
    eval_do_concat_batches=False, # this defaults to True but we'll set it to False for our evaluation function
    # save_safetensors=False # turn this off to prevent potential checkpoint issues
)

In [ ]:
## creating evaluation function
### refrence [https://github.com/huggingface/transformers/blob/336dc69d63d56f232a183a3e7f52790429b871ef/examples/pytorch/object-detection/run_object_detection.py#L160]
from typing import Tuple
## creating and function that take that "yolo" format boxes in "xyxy" format
def convert_xyxy_format(boxes: torch.Tensor, image_size: Tuple[int, int]) -> torch.Tensor:
  boxes = ops.box_convert(boxes,in_fmt="cxcywh",out_fmt="xyxy") # convert from yolo to xyxy
  height,width = image_size
  boxes = boxes * torch.tensor([[width,height,width,height]])
  return boxes

In [ ]:
# Create an evaluation function to test our model's performance
import numpy as np

from typing import Optional, Mapping

from transformers import EvalPrediction

from torchmetrics.detection.mean_ap import MeanAveragePrecision

# 1. Create a dataclass to hold our model's outputs
@dataclass
class ModelOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor

# 2. Create a compute_metrics function which takes in EvalPrediction and other required parameters
@torch.no_grad()
def compute_metrics(
    evaluation_results: EvalPrediction, # these come out of the Trainer.evaluate method, see: https://huggingface.co/docs/transformers/en/internal/trainer_utils#transformers.EvalPrediction
    image_processor: AutoImageProcessor,
    threshold: float = 0.0,
    id2label: Optional[Mapping[int, str]] = None,
) -> Mapping[str, float]:
    """
    Compute mean average mAP, mAR and their variants for the object detection task.

    Args:
        evaluation_results (EvalPrediction): Predictions and targets from evaluation.
        threshold (float, optional): Threshold to filter predicted boxes by confidence. Defaults to 0.0.
        id2label (Optional[dict], optional): Mapping from class id to class name. Defaults to None.

    Returns:
        Mapping[str, float]: Metrics in a form of dictionary {<metric_name>: <metric_value>}
    """

    # 3. Extract predictions and targets from EvalPrediction
    predictions, targets = evaluation_results.predictions, evaluation_results.label_ids

    # For metric computation we need to provide to MeanAveragePrecision
    #  - 'targets' in a form of list of dictionaries with keys "boxes", "labels"
    #  - 'predictions' in a form of list of dictionaries with keys "boxes", "scores", "labels"

    # 4. Get a list of image sizes, processed targets and processed predictions
    image_sizes = []
    post_processed_targets = []
    post_processed_predictions = []

    ### Target collection ###

    # 5. Collect target attributes in the required format for metric computation
    for batch in targets:
        # Collect ground truth image sizes, we will need them for predictions post processing
        batch_image_sizes = torch.tensor(np.array([x["orig_size"] for x in batch])) # turn into a list of numpy arrays first, then tensors
        image_sizes.append(batch_image_sizes)

        # Collect targets in the required format for metric computation
        # boxes were converted to YOLO format needed for model training
        # here we will convert them to Pascal VOC format (x_min, y_min, x_max, y_max)
        # or XYXY format. We do this because the boxes out of preprocess() are in
        # CXCYWH normalized format.
        for image_target in batch:

            # Get boxes and convert from CXCYWH to XYXY
            boxes = torch.tensor(image_target["boxes"])
            boxes = convert_xyxy_format(boxes=boxes, image_size=image_target["orig_size"])

            # Get labels
            labels = torch.tensor(image_target["class_labels"])

            # Append box and label pairs in format requried for MeanAveragePrecision class
            post_processed_targets.append({"boxes": boxes,
                                           "labels": labels})

    ### Prediction collection ###

    # 6. Collect predictions in the required format for metric computation,
    # model produce boxes in YOLO format (CXCYWH), then image_processor.post_process_object_detection to
    # convert them to Pascal VOC format (XYXY).
    for pred_batch, target_sizes in zip(predictions, image_sizes):

        # pred_batch comes in the form of a tuple: (loss, logits, boxes)
        pred_batch_loss, pred_batch_logits, pred_batch_boxes = pred_batch[0], pred_batch[1], pred_batch[2]

        model_output = ModelOutput(logits=torch.tensor(pred_batch_logits),
                                   pred_boxes=torch.tensor(pred_batch_boxes))

        # Post process the model outputs
        post_processed_output = image_processor.post_process_object_detection(
                                                    outputs=model_output,
                                                    threshold=threshold,
                                                    target_sizes=target_sizes) # target sizes required to shape boxes in correct ratio of original image

        # Extend post_processed_output in form `[{"boxes": [...], "labels": [...], "scores": [...]}]`
        # We extend because post_process_object_detection returns a list of dictionaries, so rather than append a list to a list we extend the existing list
        post_processed_predictions.extend(post_processed_output)

    # 7. Compute mAP
    max_detection_thresholds = [1, 10, 100] # 1 = mar@1, mar@10, mar@100 (100 = default max total boxes for post processed predictions out of object detection model)
    metric = MeanAveragePrecision(box_format="xyxy",
                                  class_metrics=True,
                                  max_detection_thresholds=max_detection_thresholds)
    metric.warn_on_many_detections = False # don't output a warning when large amount of detections come out (the sorting handles this anyway)
    metric.update(post_processed_predictions,
                  post_processed_targets)
    metrics = metric.compute()

    # Optional: print metrics dict for troubleshooting
    # print(metrics)

    # 8. Extract list of per class metrics with separate metric for each class
    classes = metrics.pop("classes")
    map_per_class = metrics.pop("map_per_class")

    # Optional: mAR@N per class (mAR = Mean Average Recall)
    mar_per_class = metrics.pop("mar_100_per_class")

    # 9. Prepare metrics per class in the form of a dict with metric names -> values, e.g. {"metric_name": 42.0, ...}
    # for class_id, class_map in zip(classes, map_per_class):
    for class_id, class_map, class_mar in zip(classes, map_per_class, mar_per_class):
        class_name = id2label[class_id.item()] if id2label is not None else class_id.item()
        metrics[f"map_{class_name}"] = class_map

        # Optional: mAR@100 per class
        metrics[f"mar_100_{class_name}"] = class_mar

    # 10. Round metrics for suitable visual output
    metrics = {k: round(v.item(), 4) for k, v in metrics.items()}

    # Optional: print metrics dict for troubleshooting
    # print(metrics)

    return metrics

# 11. Create a partial function for our compute_metrics function (we'll pass this to compute_metrics in Trainer)
eval_compute_metrics_fn = partial(
        compute_metrics,
        image_processor=image_processor,
        threshold=0.0,
        id2label=id2label,
)

In [ ]:
## training the model
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collation_function,
    train_dataset=processed_dataset["train"], # pass in the already preprocessed data
    eval_dataset=processed_dataset["validation"],
    compute_metrics=eval_compute_metrics_fn,
)

In [ ]:
trainer.train() # training(finetuning) the model

In [ ]:
def half_boxes(bboxes):
  if isinstance(bboxes,list):
    for boxes in bboxes:
      if isinstance(boxes,list):
        return [[coordinate // 2 for coordinate in box] for box in bboxes]
      else:
        return [coordinate // 2 for coordinate in bboxes]
  if isinstance(bboxes, np.ndarray):
        return (bboxes // 2)

  if isinstance(bboxes, torch.Tensor):
        return (bboxes // 2)

import PIL
def half_image(image:PIL.Image) -> PIL.Image:
  return image.resize(size=(image.size[0]//2,image.size[1]//2))

In [ ]:
# 7. Plotting our model's loss curves
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

# Exctract loss values
train_loss = [item["loss"] for item in log_history if "loss" in item]
eval_loss = [item["eval_loss"] for item in log_history if "eval_loss" in item]

# Extract mAP values
eval_map = [item["eval_map"] for item in log_history if "eval_map" in item]

# Plot loss curves and mAP
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20, 7))
ax[0].plot(train_loss, label="Train loss")
ax[0].plot(eval_loss, label="Eval loss")
ax[0].set_title("Loss Curves (lower is better)")
ax[0].set_ylabel("Loss Value")
ax[0].set_xlabel("Epochs")
ax[0].legend()

ax[1].plot(eval_map, label="Eval mAP")
ax[1].set_title("Eval mAP (higher is better)")
ax[1].set_ylabel("mAP (Mean Average Precision)")
ax[1].set_xlabel("Epochs")
ax[1].legend();

In [ ]:
import time
from torchvision.transforms.functional import to_pil_image, pil_to_tensor
from torchvision.utils import draw_bounding_boxes

# Get a random sample from the test preds
random_test_pred_index = random.randint(0, len(processed_dataset["test"]))
print(f"[INFO] Making predictions on test item with index: {random_test_pred_index}")

# Get a random sample from the processed dataset
random_test_sample = processed_dataset["test"][random_test_pred_index]

# Do a single forward pass with the model (we'll time how long it takes for fun)
start_pred_time = time.time()
random_test_sample_outputs = model(pixel_values=random_test_sample["pixel_values"].unsqueeze(0).to("cuda"), # model expects input [batch_size, color_channels, height, width]
                                   pixel_mask=None)
end_pred_time = time.time()
print(f"[INFO] Total time to perform prediction: {round(end_pred_time - start_pred_time, 3)} seconds.")

# Post process a random item from test preds
random_test_sample_outputs_post_processed = image_processor.post_process_object_detection(
    outputs=random_test_sample_outputs,
    threshold=0.10, # prediction probability threshold for boxes (note: boxes from an untrained model will likely be bad)
    target_sizes=random_test_sample["labels"]["orig_size"].unsqueeze(0) # original input image size (or whichever target size you'd like), required to be same number of input items in a list
)

# Extract scores, labels and boxes
random_test_sample_pred_scores = random_test_sample_outputs_post_processed[0]["scores"]
random_test_sample_pred_labels = random_test_sample_outputs_post_processed[0]["labels"]
random_test_sample_pred_boxes = half_boxes(random_test_sample_outputs_post_processed[0]["boxes"])
print(f'[INFO] scores \n{random_test_sample_pred_scores}')
print(f'[INFO] labels \n{random_test_sample_pred_labels}')
print(f'[INFO] boxes \n{random_test_sample_pred_boxes}')

# Create a list of labels and colours to plot on the boxes
random_test_sample_pred_to_score_tuples = [(id2label[label_pred.item()], round(score_pred.item(), 4))
                                           for label_pred, score_pred in zip(random_test_sample_pred_labels, random_test_sample_pred_scores)]
random_test_sample_labels_to_plot = [f"Pred: {item[0]} ({item[1]})" for item in random_test_sample_pred_to_score_tuples]
random_test_sample_colours_to_plot = [color_palette[item[0]] for item in random_test_sample_pred_to_score_tuples]

print(f"[INFO] Labels with scores:")
for label in random_test_sample_labels_to_plot:
    print(label)

# Plot the predicted boxes on the random test image
test_pred_box_image = to_pil_image(
    pic=draw_bounding_boxes(
        image=pil_to_tensor(pic=half_image(dataset["test"][random_test_pred_index]["image"])),
        boxes=random_test_sample_pred_boxes,
        colors=random_test_sample_colours_to_plot,
        labels=random_test_sample_labels_to_plot,
        width=3
    )
)

test_pred_box_image

In [ ]:
# Get ground truth image
# Get ground truth image
ground_truth_image = half_image(dataset["test"][random_test_pred_index]["image"])

# 1. Convert and stack boxes directly (Remove ops.box_convert and unsqueeze(0))
ground_truth_boxes = [convert_xyxy_format(boxes=input_box, image_size=random_test_sample["labels"]["orig_size"]) for input_box in random_test_sample["labels"]["boxes"]]
ground_truth_boxes = torch.stack(half_boxes(ground_truth_boxes)) # Shape: [N, 4]

# Get ground truth labels and colours
ground_truth_labels = [id2label[label.item()] for label in random_test_sample["labels"]["class_labels"]]
ground_truth_colours = [color_palette[label] for label in ground_truth_labels]

# Create ground truth box plot image
test_ground_truth_box_image = to_pil_image(
    pic=draw_bounding_boxes(
        image=pil_to_tensor(pic=ground_truth_image),
        boxes=ground_truth_boxes,
        colors=ground_truth_colours,
        labels=ground_truth_labels,
        width=3
    )
)

# Plot ground truth image and boxes to predicted image and boxes
fig, ax = plt.subplots(ncols=2, figsize=(16, 10))
ax[0].imshow(test_ground_truth_box_image)
ax[0].set_title("Ground Truth Image and Boxes")
ax[0].axis(False)
ax[1].imshow(test_pred_box_image)
ax[1].set_title("Predicted Boxes")
ax[1].axis(False)

plt.show()

In [ ]:
# Save the model
from datetime import datetime # optional: add a date of when we trained our model

# Get details to add to model's save path
training_epochs_ = training_args.num_train_epochs
learning_rate_ = "{:.0e}".format(training_args.learning_rate)

# Create model save path with some training details
model_save_path = f"models/learn_hf_rt_detrv2_finetuned_trashify_no_aug_{training_epochs_}_epochs_lr_{learning_rate_}"

# Save model to file
print(f"[INFO] Saving model to: {model_save_path}")
model.save_pretrained(model_save_path)

In [ ]:
# Make sure trainer has the processor class (this can sometimes be automatically assigned, however, we'll hard code it just to be safe)
model.processing_class = image_processor

In [ ]:
model_to_hub = model.push_to_hub(
    "RahulKate-173/rt_detrv2_finetuned_trashify_box_detector_v2",
   # token 
)

In [ ]:
processor_push_to_hub = image_processor.push_to_hub(
    "RahulKate-173/rt_detrv2_finetuned_trashify_box_detector_v2",
    
)